# Autoportrait dans le rap — Qwen3-32B

Ce carnet a un seul objectif : **annoter les chansons de `Rap/corpus.csv` avec Qwen/Qwen3-32B en mode raisonnement**. Le chemin du fichier, depuis le dossier de ce carnet (`Rap/Autoportrait/`), est `../corpus.csv`. Un paramètre unique choisit le périmètre : `taille_echantillon = 100` actuellement, un autre entier positif pour un autre effectif, ou `None` pour toutes les chansons exploitables. Le même parcours exécute directement la sélection choisie, **sans pilote préalable ni étape d'approbation**. Aucune comparaison entre modèles n'est réalisée.

Chaque chanson reçoit une décision parmi `autoportrait`, `pas_autoportrait`, `je_ne_sais_pas`, ainsi qu'une justification. La consigne demande une justification en français, sans citation des paroles. La définition est lue dans [consigne.md](consigne.md), sans modification. Le nettoyage, les messages et le schéma JSON sont réutilisés depuis [annoter.py](annoter.py) afin de garder une consigne unique. Aucun fichier de référence ni ancienne annotation n'est transmis au modèle. Les contrôles techniques vérifient le format et les valeurs, pas le respect sémantique de la consigne.

## Utilisation
1. Choisir un noyau Python avec **pandas** et **tqdm** (par exemple **Rap R05b (PyTorch CPU)**). Les dépendances figurent dans [requirements.txt](requirements.txt) ; si nécessaire, les installer avec le Python du noyau. Exécuter depuis la racine du dépôt, `Rap/` ou `Rap/Autoportrait/`.
2. **Avant de démarrer un serveur**, contrôler `nvidia-smi` : mémoire libre, activité et processus de chaque GPU. Ne pas arrêter les processus d'autres utilisateurs. Choisir un GPU libre avec `CUDA_VISIBLE_DEVICES`. Le noyau du carnet reste un client CPU ; le modèle est chargé par le serveur vLLM, pas par ce carnet.
3. Disposer de **deux instances indépendantes de vLLM / Qwen/Qwen3-32B**, avec `--reasoning-parser qwen3` : GPU physique 1 sur `http://127.0.0.1:8016/v1` et GPU physique 2 sur `http://127.0.0.1:8017/v1`. Chaque GPU porte un modèle complet ; il ne s'agit pas de tensor parallelism. Le carnet ne démarre ni n'arrête les serveurs et ne télécharge pas de modèle. `QWEN_BASE_URLS` accepte une liste d'URL séparées par des virgules ; à défaut, `QWEN_BASE_URL` permet de choisir une seule adresse. `QWEN_API_KEY` fournit une clé commune aux serveurs.
4. Choisir `taille_echantillon` dans la configuration : **100 actuellement**, un entier positif pour un autre effectif, ou **`None` pour le corpus entier**. Une taille invalide ou supérieure au corpus est refusée ; elle n'est jamais remplacée silencieusement par une autre sélection.
5. **Après cette mise à jour à deux GPU, réexécuter toutes les cellules depuis la configuration**, afin de recréer la file de créneaux et de remplacer les anciennes fonctions dans le noyau. La dernière cellule annote directement la sélection choisie, puis affiche ses résultats. Avec `None`, elle lance donc le corpus entier : vérifier le paramètre avant de l'exécuter. Consulter ensuite les décisions, justifications et raisonnements dans les exports. Une relance reprend les réussites compatibles et retente les erreurs. Ne pas modifier la configuration pendant qu'une annotation est en cours.

Les résultats vont dans `data/results/`, les métriques dans `data/performance/`, sous ce dossier. Les exports se nomment `autoportrait_qwen_annotation_{n}.csv` et `.json`, où `n` est le nombre de chansons sélectionnées (100 actuellement). Un nouveau lot de même taille remplace ces exports ; les journaux JSONL restent distincts par empreinte. Ils sont sauvegardés après chaque réponse ; une relance reprend les réussites et retente les erreurs. **Une erreur réseau, HTTP ou JSON n'est jamais une décision `je_ne_sais_pas`.**

## Paramètres et limites

Ne pas confondre les tailles : **`taille_echantillon=100` choisit le total de chansons** ; **`taille_lot=128` limite les tâches soumises à la fois** ; **`concurrence_par_serveur=8` limite les requêtes par instance**. La concurrence totale est calculée automatiquement : **16 workers pour deux serveurs**. Une file de créneaux partagée attribue chaque chanson à un serveur disposant d'une place ; le créneau est rendu même en cas d'erreur. Un serveur rapide peut ainsi traiter davantage de chansons, sans partage rigide en deux moitiés. Une tentative supplémentaire reste sur le même serveur. Un seul client centralise le journal et les exports, dans l'ordre du corpus. La colonne `serveur` permet d'identifier l'instance utilisée ; les métriques enregistrent les adresses et la concurrence. Aucun basculement silencieux n'est effectué si un serveur manque : tous sont vérifiés avant une nouvelle inférence.

Le raisonnement Qwen3 est activé (`enable_thinking=True`), avec une température de 0 et une graine de 42. Ces réglages facilitent la reproductibilité, sans garantir une identité absolue entre versions de serveur ou matériels. Le budget de **8192 tokens** couvre **le raisonnement et la réponse finale réunis** ; une génération coupée n'est jamais acceptée. Le délai de 600 secondes s'applique à chaque requête, avec deux tentatives au maximum ; ce n'est pas une limite de durée pour toute la sélection.

Le contexte serveur doit accueillir les messages d'entrée et le budget de sortie. Un contexte de 16384 tokens ne garantit pas que toute chanson longue y tiendra. Les paroles sont envoyées intégralement après nettoyage : aucun filtre supplémentaire, aucune troncature silencieuse. Un dépassement de contexte est enregistré comme erreur à corriger, pas comme une hésitation du modèle. La mémoire GPU et l'espace disque disponible restent à surveiller lors d'un traitement complet.

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
from queue import Queue
import fcntl
import hashlib
import importlib.util
import json
import os
import time
import urllib.error
import urllib.request

import pandas as pd
from tqdm.auto import tqdm

# Accepte un lancement depuis la racine du dépôt, Rap ou Autoportrait.
candidats = [Path.cwd(), Path.cwd() / 'Autoportrait', Path.cwd() / 'Rap' / 'Autoportrait']
dossiers = [p.resolve() for p in candidats if (p / 'consigne.md').is_file() and (p / 'annoter.py').is_file()]
if len(dossiers) != 1:
    raise RuntimeError('Lancer le carnet depuis la racine du dépôt, Rap ou Rap/Autoportrait.')
dossier = dossiers[0]
corpus_path = dossier.parent / 'corpus.csv'
consigne_path = dossier / 'consigne.md'
resultats_dir = dossier / 'data' / 'results'
performance_dir = dossier / 'data' / 'performance'

urls_configurees = os.environ.get('QWEN_BASE_URLS', os.environ.get(
    'QWEN_BASE_URL', 'http://127.0.0.1:8016/v1,http://127.0.0.1:8017/v1'))
base_urls = [url.strip().rstrip('/') for url in urls_configurees.split(',')]
if any(not url.startswith(('http://', 'https://')) for url in base_urls) or len(set(base_urls)) != len(base_urls):
    raise ValueError('Configurer des URL HTTP(S) distinctes dans QWEN_BASE_URLS.')
modele = 'Qwen/Qwen3-32B'
cle_api = os.environ.get('QWEN_API_KEY', 'EMPTY')
taille_echantillon = 500
graine = 42
concurrence_par_serveur = 12
concurrence = concurrence_par_serveur * len(base_urls)
serveurs_disponibles = Queue()
for _ in range(concurrence_par_serveur):
    for url in base_urls:
        serveurs_disponibles.put(url)
taille_lot = 64
timeout_secondes = 600
max_tokens = 8192
enable_thinking = True
max_essais = 2

if taille_echantillon is not None and (type(taille_echantillon) is not int or taille_echantillon <= 0):
    raise ValueError('taille_echantillon doit être un entier positif ou None pour le corpus entier.')
if min(taille_lot, concurrence, timeout_secondes, max_tokens, max_essais) <= 0:
    raise ValueError('Les tailles, délais et nombres de tentatives doivent être positifs.')
spec = importlib.util.spec_from_file_location('autoportrait_source', dossier / 'annoter.py')
if spec is None or spec.loader is None:
    raise ImportError('Impossible de charger annoter.py.')
source = importlib.util.module_from_spec(spec)
spec.loader.exec_module(source)
consigne = consigne_path.read_text(encoding='utf-8').strip()
if not consigne:
    raise ValueError('La consigne est vide.')
print('Dossier :', dossier)
print('Modèle :', modele, '| endpoints :', base_urls, '| raisonnement :', enable_thinking)
print('Workers :', concurrence, '| maximum par serveur :', concurrence_par_serveur)


## 1. Charger et préparer le corpus complet

### Fichier d'entrée et identifiants

La seule source de chansons est **[Rap/corpus.csv](../corpus.csv)**, lu directement avec la fonction commune de [annoter.py](annoter.py). Le code construit son chemin depuis le dossier du carnet, et non depuis un chemin relatif dépendant du répertoire d'exécution. Les colonnes attendues sont `artist`, `title`, `year` et `lyrics`. `source_row_index` conserve la **position de l'enregistrement de données, à partir de zéro**, dans ce fichier. Ce n'est ni un numéro de ligne physique (les paroles peuvent contenir des retours à la ligne), ni un `document_id`. Cet identifiant permet de rattacher chaque annotation à l'enregistrement original ; un corpus réordonné peut changer les identifiants et l'empreinte de reprise.

« Corpus complet » désigne ici **toutes les lignes avec des paroles exploitables**, sans restriction sur le titre, l'artiste, l'année ou le nombre de mots. Comme dans le script commun, les paroles absentes, vides ou égales à `NA` ne sont pas retenues. Le nettoyage retire les résidus déjà gérés par ce script, sans tronquer les textes. Les chansons devenues vides après nettoyage sont également exclues et leur nombre est affiché explicitement. Si aucun texte exploitable ne reste, l'exécution s'arrête.

### Ce qui est conservé ou envoyé

Les chansons restent dans l'ordre du corpus. Les métadonnées exportées sont la position source, l'artiste, le titre, l'année et le nombre de mots après nettoyage. Aucune ancienne annotation n'est utilisée pour préparer les textes ou les messages ; seuls les journaux compatibles pourront ensuite être relus pour la reprise. Le modèle reçoit uniquement la consigne et les paroles nettoyées. L'aperçu ci-dessous n'affiche pas les paroles. **Même pour l'échantillon actuel de 100 chansons, cette étape lit et prépare le corpus exploitable entier en mémoire**, puis le tirage a lieu dans la dernière cellule. La barre de préparation commence après la lecture du CSV ; elle ne mesure donc pas cette lecture initiale. **Aucune requête d'annotation n'est envoyée à cette étape.**

In [ ]:
def charger_chansons():
    corpus = source.lire_corpus(corpus_path, None)
    chansons = []
    vides = 0
    for position, chanson in tqdm(corpus.items(), total=len(corpus), desc='Préparation du corpus', unit='chansons'):
        paroles = source.nettoyer(chanson['paroles'])
        if not paroles:
            vides += 1
            continue
        chansons.append({**chanson, 'source_row_index': position, 'paroles': paroles,
                         'mots_nettoyes': len(paroles.split())})
    print(f'{len(chansons)} chansons prêtes ; {vides} textes supplémentaires vides après nettoyage.')
    if not chansons:
        raise ValueError('Aucune chanson avec des paroles exploitables dans le corpus.')
    return chansons

chansons = charger_chansons()
display(pd.DataFrame(chansons).drop(columns='paroles').head())
print('Aucune inférence lancée ; aucun texte tronqué.')


## 2. Client vLLM et validation stricte

### Contrat serveur et réponse finale

Avant toute nouvelle inférence, **chaque serveur configuré** doit annoncer exactement `Qwen/Qwen3-32B` sur son endpoint `/v1/models`. Si une instance est inaccessible ou sert un autre modèle, le traitement s'arrête avant l'envoi des chansons ; il ne continue pas silencieusement sur un seul GPU. Les deux instances doivent utiliser les mêmes poids, le même parseur de raisonnement et les mêmes réglages de génération ; le contrôle du nom du modèle ne suffit pas à certifier cette identité. Chaque requête active le raisonnement dans le gabarit de conversation et impose le schéma JSON de [annoter.py](annoter.py) à la réponse finale.

Une réponse réussie doit avoir une génération terminée (`finish_reason='stop'`), un objet JSON avec **exactement** `etiquette` et `justification`, une étiquette autorisée et une justification textuelle non vide. En mode raisonnement, elle doit également comporter un champ séparé `reasoning` ou `reasoning_content` non vide. Le parseur serveur `--reasoning-parser qwen3` assure cette séparation : le raisonnement n'est pas extrait approximativement du JSON final. Il est conservé dans la colonne `raisonnement`, distincte de la justification courte.

### Erreurs et tentatives

Les erreurs réseau, réponses mal formées et erreurs HTTP transitoires (408, 429, 500, 502, 503, 504) sont retentées dans la limite de `max_essais`, avec une attente entre les tentatives. Une erreur HTTP permanente, notamment un contexte trop court, n'est pas retentée immédiatement. Après échec, la chanson conserve `statut='erreur'`, un message explicite et une étiquette vide. **`je_ne_sais_pas` reste une décision du modèle, jamais un substitut à une panne.** Une réponse syntaxiquement valide ne garantit pas la qualité de la décision : celle-ci relève de la revue humaine.

In [ ]:
ouvreur = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def appeler_api(route, corps=None, *, serveur):
    requete = urllib.request.Request(
        serveur + route,
        data=None if corps is None else json.dumps(corps).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {cle_api}'},
    )
    with ouvreur.open(requete, timeout=timeout_secondes) as reponse:
        return json.loads(reponse.read().decode('utf-8'))

def verifier_endpoint(serveur):
    try:
        reponse = appeler_api('/models', serveur=serveur)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f'Serveur vLLM : HTTP {exc.code} sur {serveur}/models. Vérifier la configuration et la clé API.') from exc
    except (urllib.error.URLError, OSError) as exc:
        raise RuntimeError(
            f'Serveur Qwen injoignable sur {serveur}. Démarrer vLLM avec --reasoning-parser qwen3 '
            'sur un GPU libre, attendre sa disponibilité, puis relancer la cellule. '
            'Le notebook ne démarre pas le serveur automatiquement.') from exc
    modeles = [item['id'] for item in reponse['data']]
    if modele not in modeles:
        raise RuntimeError(f'{modele} non servi sur {serveur}. Modèles disponibles : {modeles}')
    print('Serveur disponible :', serveur, '|', modele)

def verifier_serveur():
    for serveur in base_urls:
        verifier_endpoint(serveur)

def valider_annotation(objet, raisonnement):
    if enable_thinking and (not isinstance(raisonnement, str) or not raisonnement.strip()):
        raise ValueError('Raisonnement absent : vérifier enable_thinking et --reasoning-parser qwen3 côté serveur.')
    if not isinstance(objet, dict) or set(objet) != {'justification', 'etiquette'}:
        raise ValueError('Deux clés exactes attendues : justification et etiquette.')
    if objet['etiquette'] not in source.ETIQUETTES:
        raise ValueError(f"Étiquette inconnue : {objet['etiquette']!r}")
    if not isinstance(objet['justification'], str) or not objet['justification'].strip():
        raise ValueError('Justification vide ou non textuelle.')
    return {'justification': objet['justification'].strip(), 'etiquette': objet['etiquette'],
            'raisonnement': raisonnement.strip() if isinstance(raisonnement, str) else ''}

def valider_reponse(reponse):
    choix = reponse['choices'][0]
    if choix.get('finish_reason') != 'stop':
        raise ValueError(f"Génération incomplète : {choix.get('finish_reason')!r}")
    message = choix['message']
    if not isinstance(message['content'], str):
        raise ValueError('Contenu absent ou non textuel.')
    return valider_annotation(json.loads(message['content']),
                              message.get('reasoning') or message.get('reasoning_content'))

def annoter_chanson(chanson):
    serveur = serveurs_disponibles.get()
    try:
        return {**annoter_sur_serveur(chanson, serveur), 'serveur': serveur}
    finally:
        serveurs_disponibles.put(serveur)

def annoter_sur_serveur(chanson, serveur):
    debut = time.perf_counter()
    corps = {
        'model': modele,
        'messages': source.messages(consigne, chanson['paroles'], False),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'chat_template_kwargs': {'enable_thinking': enable_thinking},
        **source.CONTRAINTES['json_schema'],
    }
    for essai in range(1, max_essais + 1):
        try:
            reponse = appeler_api('/chat/completions', corps, serveur=serveur)
            annotation = valider_reponse(reponse)
            return {**annotation, 'statut': 'ok', 'erreur': '', 'essais': essai,
                    'secondes': time.perf_counter() - debut}
        except urllib.error.HTTPError as exc:
            erreur = f'HTTP {exc.code} : {exc.read().decode("utf-8", "replace")[:1000]}'
            if exc.code not in {408, 429, 500, 502, 503, 504}:
                break
        except (OSError, ValueError, KeyError, IndexError, TypeError) as exc:
            erreur = f'{type(exc).__name__} : {exc}'
        if essai < max_essais:
            time.sleep(2 * essai)
    return {'justification': '', 'etiquette': '', 'raisonnement': '', 'statut': 'erreur', 'erreur': erreur,
            'essais': essai, 'secondes': time.perf_counter() - debut}


## 3. Journal, reprise, progression et exports

### Reprendre sans mélanger les expériences

Une empreinte SHA-256 couvre le modèle, la liste des URL, la consigne, le script commun, les paramètres de génération et les chansons sélectionnées. La version multi-serveurs utilise de nouveaux journaux : les anciens résultats ne sont pas effacés ni repris comme un benchmark à deux GPU. La concurrence, la taille des lots techniques et les délais ne font pas partie de cette empreinte : les ajuster ne force pas à réannoter les réussites. Les noms techniques des journaux conservent le mode, le périmètre, la taille et l'empreinte ; ils ne sont pas les noms simplifiés des exports destinés à la lecture. Cette empreinte ne certifie pas la version du serveur ni ses poids effectifs.

Chaque réponse, y compris une erreur, est écrite et synchronisée sur disque dès sa réception. À la relance, les réussites compatibles sont vérifiées puis reprises ; seules les chansons absentes ou en erreur sont soumises au serveur. Un journal endommagé ou incompatible provoque une erreur explicite et n'est pas ignoré. Si tout est déjà réussi, les exports peuvent être régénérés **sans serveur et sans GPU**. Un verrou sur le nom de sortie empêche deux lots de même taille d'écrire simultanément les mêmes exports.

### Sorties et ordre des lignes

À la fin du traitement, les fichiers lisibles sont `data/results/autoportrait_qwen_annotation_{n}.csv` et `.json`, sous `Rap/Autoportrait/`, avec une ligne par chanson sélectionnée, **erreurs comprises**, dans l'ordre du corpus. **`n` est la taille totale de la sélection**, pas le nombre de réussites ni `taille_lot`. Le fichier `_erreurs.csv` isole les échecs et reste présent même s'il ne contient que les en-têtes. Une autre sélection de même taille remplace ces exports lisibles ; les journaux distincts permettent de retrouver les résultats de chaque expérience. Les paroles ne sont pas recopiées dans les exports, mais le raisonnement peut en reprendre des éléments : ces fichiers restent des données locales à protéger.

### Progression et performances

La barre affiche l'avancement, le nombre d'erreurs, le débit en chansons/s et une ETA estimée à partir de la progression observée ; cette ETA n'est pas une durée garantie. Le temps et le débit sont **mesurés côté client**, pas sur le GPU. Les métriques CSV, rangées dans `data/performance/`, distinguent les réussites reprises des chansons traitées pendant la session. Une reprise complète n'est donc pas un nouveau benchmark d'inférence. Une relance remplace la synthèse du même journal par celle de la nouvelle session, mais **ajoute une ligne à l'historique cumulatif `perfGPU_autoportrait_{n}.csv`**, sans effacer les précédentes.

L'historique contient la date UTC, la durée client, `secondes_par_chanson_session`, `chansons_s_session`, les réussites, erreurs et reprises, le nombre et les adresses des serveurs, la concurrence et les paramètres de génération. `type_execution` distingue `calcul` (aucune reprise), `mixte` et `reprise_cache` (aucune chanson nouvellement traitée). Les secondes par chanson valent la durée totale divisée par le nombre de chansons traitées pendant la session, erreurs comprises ; elles restent vides si tout provient du cache. **Ce ratio global s/chanson n'est pas la latence moyenne d'une requête**, puisque plusieurs requêtes tournent simultanément. Malgré le nom `perfGPU`, aucune durée GPU n'est mesurée. Les distributions d'étiquettes excluent les erreurs techniques ; aucune précision, mémoire GPU ou émission carbone n'est déduite de ces mesures.

En cas d'interruption, les réponses déjà journalisées restent disponibles. Les tâches en attente sont annulées lorsque possible ; les requêtes déjà parties peuvent finir avant que la cellule rende la main. **Les exports CSV/JSON ne sont pas actualisés après chaque réponse** : avant la fin du traitement, ils peuvent être absents ou correspondre à une exécution antérieure. Le journal JSONL est la source de reprise pendant le calcul. Relancer le même périmètre pour poursuivre et régénérer les exports finaux.

In [ ]:
def empreinte(selection):
    configuration = {
        'version_pipeline': 3, 'modele': modele, 'urls': base_urls, 'consigne': consigne,
        'annoter_sha256': hashlib.sha256((dossier / 'annoter.py').read_bytes()).hexdigest(),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'enable_thinking': enable_thinking, 'contrainte': source.CONTRAINTES['json_schema'],
        'selection': selection,
    }
    return hashlib.sha256(json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def executer_annotation(selection, perimetre):
    if not selection:
        raise ValueError('Sélection vide.')
    resultats_dir.mkdir(parents=True, exist_ok=True)
    performance_dir.mkdir(parents=True, exist_ok=True)
    signature = empreinte(selection)
    mode = 'reason' if enable_thinking else 'sans_reason'
    prefixe = f'autoportrait_qwen32_{mode}_{perimetre}_n{len(selection)}_{signature[:12]}'
    prefixe_sortie = f'autoportrait_qwen_annotation_{len(selection)}'
    journal = resultats_dir / f'{prefixe}.jsonl'
    ids = [c['source_row_index'] for c in selection]
    ids_selection = set(ids)
    if len(ids) != len(ids_selection):
        raise ValueError('Sélection avec doublons.')
    debut = time.perf_counter()
    with (resultats_dir / f'{prefixe_sortie}.lock').open('a') as verrou:
        try:
            fcntl.flock(verrou, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError('Un lot de même taille écrit déjà ces fichiers de sortie.') from exc
        resultats = {}
        if journal.exists():
            with journal.open(encoding='utf-8') as fichier:
                for numero, ligne in enumerate(fichier, 1):
                    try:
                        resultat = json.loads(ligne)
                    except json.JSONDecodeError as exc:
                        raise ValueError(f'Journal endommagé : {journal}, ligne {numero}. Réparer avant reprise.') from exc
                    if resultat['empreinte'] != signature or resultat['source_row_index'] not in ids_selection:
                        raise ValueError('Journal incompatible avec la sélection courante.')
                    if resultat['statut'] not in {'ok', 'erreur'}:
                        raise ValueError('Statut invalide dans le journal.')
                    if resultat['statut'] == 'ok':
                        valider_annotation({k: resultat[k] for k in ('justification', 'etiquette')},
                                           resultat.get('raisonnement'))
                    resultats[resultat['source_row_index']] = resultat
        restantes = [c for c in selection if resultats.get(c['source_row_index'], {}).get('statut') != 'ok']
        reprises = len(selection) - len(restantes)
        print(f'{perimetre} : {reprises} réussites reprises ; {len(restantes)} chansons à traiter.')
        if restantes:
            verifier_serveur()
        traitees = erreurs = 0
        with journal.open('a', encoding='utf-8') as fichier:
            with ThreadPoolExecutor(max_workers=concurrence) as executeur:
                taches = {}
                try:
                    with tqdm(total=len(selection), initial=reprises, desc=f'Annotation {perimetre}',
                              unit='chansons', mininterval=0.5) as barre:
                        for debut_lot in range(0, len(restantes), taille_lot):
                            lot = restantes[debut_lot:debut_lot + taille_lot]
                            taches = {executeur.submit(annoter_chanson, c): c for c in lot}
                            for tache in as_completed(taches):
                                chanson = taches[tache]
                                resultat = {**{k: v for k, v in chanson.items() if k != 'paroles'},
                                            **tache.result(), 'modele': modele, 'empreinte': signature,
                                            'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
                                fichier.write(json.dumps(resultat, ensure_ascii=False) + '\n')
                                fichier.flush()
                                os.fsync(fichier.fileno())
                                resultats[chanson['source_row_index']] = resultat
                                traitees += 1
                                erreurs += resultat['statut'] == 'erreur'
                                barre.update(1)
                                barre.set_postfix(erreurs=erreurs,
                                    chansons_s=f'{traitees / max(time.perf_counter() - debut, 1e-9):.2f}', refresh=False)
                except BaseException:
                    for tache in taches:
                        tache.cancel()
                    raise
        ordonnes = [resultats[position] for position in ids]
        if len(ordonnes) != len(selection):
            raise RuntimeError('Effectif final incohérent.')
        tableau = pd.DataFrame(ordonnes)
        tableau.to_csv(resultats_dir / f'{prefixe_sortie}.csv', index=False, encoding='utf-8-sig')
        (resultats_dir / f'{prefixe_sortie}.json').write_text(
            json.dumps(ordonnes, ensure_ascii=False, indent=2), encoding='utf-8')
        tableau.loc[tableau['statut'] == 'erreur'].to_csv(
            resultats_dir / f'{prefixe_sortie}_erreurs.csv', index=False, encoding='utf-8-sig')
        secondes = time.perf_counter() - debut
        metriques = {'perimetre': perimetre, 'modele': modele, 'empreinte': signature,
                     'selection': len(selection), 'reprises': reprises, 'traitees_session': traitees,
                     'reussites': int((tableau['statut'] == 'ok').sum()),
                     'erreurs': int((tableau['statut'] == 'erreur').sum()),
                     'secondes_session_client': secondes, 'chansons_s_session': traitees / secondes,
                     'secondes_par_chanson_session': secondes / traitees if traitees else None,
                     'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                     'type_execution': 'reprise_cache' if not traitees else ('mixte' if reprises else 'calcul'),
                     'nombre_serveurs': len(base_urls),
                     'concurrence': concurrence, 'concurrence_par_serveur': concurrence_par_serveur,
                     'serveurs': json.dumps(base_urls), 'taille_lot': taille_lot, 'max_tokens': max_tokens,
                     'enable_thinking': enable_thinking,
                     'raisonnements_non_vides': int(tableau['raisonnement'].str.strip().ne('').sum())}
        pd.DataFrame([metriques]).to_csv(performance_dir / f'{prefixe}_performance.csv', index=False)
        cumul_path = performance_dir / f'perfGPU_autoportrait_{len(selection)}.csv'
        with cumul_path.open('a+', encoding='utf-8', newline='') as cumul:
            fcntl.flock(cumul, fcntl.LOCK_EX)
            cumul.seek(0, os.SEEK_END)
            pd.DataFrame([metriques]).to_csv(cumul, index=False, header=cumul.tell() == 0)
            cumul.flush()
            os.fsync(cumul.fileno())
        valides = tableau.loc[tableau['statut'] == 'ok']
        comptes = valides['etiquette'].value_counts().reindex(source.ETIQUETTES, fill_value=0)
        distribution = pd.DataFrame([
            {'etiquette': e, 'effectif': comptes[e],
             'pourcentage_valides': 100 * comptes[e] / len(valides) if len(valides) else None}
            for e in source.ETIQUETTES])
        distribution.to_csv(performance_dir / f'{prefixe}_distribution.csv', index=False)
    display(pd.DataFrame([metriques]))
    display(distribution)
    if metriques['erreurs']:
        print('ATTENTION : erreurs techniques présentes ; voir le CSV erreurs et relancer pour les retenter.')
    print('Annotations :', resultats_dir / f'{prefixe_sortie}.csv')
    print('Performances cumulées :', cumul_path)
    return tableau


## 4. Sélection et annotation du lot choisi

### Un seul paramètre pour un échantillon ou la totalité

Avec `taille_echantillon = 100`, la cellule tire **100 chansons sans remise dans le corpus exploitable**, avec la graine 42. Tout autre entier positif, inférieur ou égal au nombre de chansons exploitables, choisit cet autre effectif. Les positions tirées sont triées pour conserver l'ordre relatif du corpus. Le tirage n'est pas stratifié : un petit échantillon ne permet pas d'estimer précisément les fréquences des catégories. Le même corpus préparé, la même taille et la même graine reproduisent le tirage dans le même environnement logiciel.

Avec `taille_echantillon = None`, **toutes les chansons exploitables sont sélectionnées**, sans tirage et sans traitement préalable d'un petit lot. Il n'y a ni pilote obligatoire, ni bouton d'approbation, ni réannotation intermédiaire. Exécuter cette cellule démarre directement le périmètre choisi. Repartir de la configuration après avoir changé le paramètre ; recharger aussi les données si le corpus a changé sur disque.

### Contrôles techniques et lecture des résultats

Le contrôle `/v1/models` précède toute nouvelle requête d'annotation, sauf lors d'une reprise entièrement en cache. Chaque réussite exige un raisonnement séparé non vide et un JSON final strictement conforme. À la fin du traitement, les erreurs sont exportées et la cellule signale explicitement un lot incomplet si certaines chansons échouent. Les fichiers finaux contiennent toutes les lignes sélectionnées, erreurs comprises, dans l'ordre du corpus. Une relance traite uniquement les chansons encore absentes ou en erreur ; une reprise complète ne nécessite pas de serveur.

Les exports se nomment `autoportrait_qwen_annotation_100.csv` et `.json` pour le réglage actuel, ou `autoportrait_qwen_annotation_{n}` avec l'effectif réel du corpus quand le paramètre vaut `None`. L'aperçu est limité à 20 lignes pour ne pas saturer le carnet ; cette limite ne réduit pas les fichiers. Lire les décisions et les justifications, puis consulter `raisonnement` pour les cas ambigus. Vérifier que le modèle applique bien **AUTO + PORTRAIT**, et ne se fonde pas seulement sur la présence de « je ». Un raisonnement long et un taux de réussite technique de 100 % ne constituent pas une mesure de précision. Aucun accord avec Claude n'est calculé.

In [ ]:
if taille_echantillon is None:
    selection = chansons
    perimetre = 'corpus'
else:
    if type(taille_echantillon) is not int or not 0 < taille_echantillon <= len(chansons):
        raise ValueError('La taille doit être un entier positif ne dépassant pas le corpus, ou None.')
    indices = sorted(pd.Series(range(len(chansons))).sample(n=taille_echantillon, random_state=graine).tolist())
    selection = [chansons[i] for i in indices]
    perimetre = 'echantillon'
print(f'Périmètre : {perimetre} ; {len(selection)} chansons sélectionnées sur {len(chansons)}.')
resultats = executer_annotation(selection, perimetre)
display(resultats[['source_row_index', 'artiste', 'titre', 'etiquette', 'justification', 'erreur']].head(20))
if not resultats['statut'].eq('ok').all():
    raise RuntimeError('Lot incomplet : consulter le fichier erreurs et relancer pour reprendre.')
